**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Ajouter `ZONE_PEDO` au parcellaire brut

# Création du Fichier `typeDeSolParZH.shp` : Jointure Parcellaire et Donnees Terrain

## 1. Description du projet

Ce notebook a pour unique objectif de créer un **parcellaire enrichi**. Il part du shapefile géométrique brut, le nettoie et l'enrichit avec les attributs de classification (`Type_champ`, `ZONE_PEDO`) nécessaires pour les jointures et analyses futures.

Le fichier de sortie de ce notebook, `parcellaire_enrichi.shp`, servira de base géométrique propre pour la consolidation finale.

---
## 2. Objectifs

* Charger le shapefile du parcellaire brut et les données CSV de M. Malou.
* Appliquer les corrections de nettoyage (suppression de doublons et de parcelles problématiques).
* Enrichir le parcellaire avec l'attribut `Type_champ`.
* Créer et nettoyer la colonne de classification `ZONE_PEDO`.
* Sauvegarder le GeoDataFrame final dans un nouveau fichier shapefile "processed".

---
## 3. Fichiers en Entrée et en Sortie

### 3.1. Fichiers en Entrée
* **Parcellaire Brut :** `data/sols/shapefiles/raw/Parcellaire_Arbre_Carbone.shp`
* **Type de Champ :** `data/sols/csv/raw/malou_0_30.csv`

### 3.2. Fichier en Sortie
* **Parcellaire Enrichi :** `data/sols/shapefiles/processed/parcellaire_enrichi.shp`
---


In [3]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np

In [4]:
# --- 1. CHEMINS ET CHARGEMENT ---
base_dir = Path.cwd().parent.resolve()
input_parcellaire_path = base_dir / "data" / "sols" / "shapefiles" / "raw" / "Parcellaire_Arbre_Carbone.shp"
malou_csv_path = base_dir / "data" / "sols" / "csv" / "raw" / "malou_0_30.csv"
output_parcellaire_enrichi_path = base_dir / "data" / "sols" / "shapefiles" / "processed" / "parcellaire_enrichi.shp"

gdf_parcellaire = gpd.read_file(input_parcellaire_path)
df_malou = pd.read_csv(malou_csv_path)
print("✅ Fichiers bruts chargés.")

✅ Fichiers bruts chargés.


In [5]:
# --- 2. NETTOYAGE DES SOURCES ---
# Nettoyer df_malou
df_malou.rename(columns={'N°_PARCELL': 'N°_PARCEL'}, inplace=True)
df_malou.drop_duplicates(subset=['N°_PARCEL'], keep='first', inplace=True)
# Nettoyer gdf_parcellaire (cohérence avec le notebook 02b)
gdf_parcellaire = gdf_parcellaire[gdf_parcellaire['N°_PARCEL'] != 201].copy()
print("✅ Nettoyage des données sources effectué.")

✅ Nettoyage des données sources effectué.


In [6]:
# --- 3. ENRICHISSEMENT AVEC 'Type_champ' ---
gdf_parcellaire = gdf_parcellaire.merge(
    df_malou[['N°_PARCEL', 'Type_champ']],
    on='N°_PARCEL',
    how='left'
)
print("✅ Parcellaire enrichi avec 'Type_champ'.")

✅ Parcellaire enrichi avec 'Type_champ'.


In [7]:
# --- 4. CRÉATION DE 'ZONE_PEDO' ---
presence_arbre = np.where(gdf_parcellaire['Arbre'] == 1, 'avec_arbr', 'sans_arbr')
gdf_parcellaire['ZONE_PEDO'] = (
    gdf_parcellaire['TYP_SOL'].astype(str) + '_' +
    gdf_parcellaire['Type_champ'].astype(str) + '_' +
    presence_arbre
).str.lower()
gdf_parcellaire['ZONE_PEDO'] = gdf_parcellaire['ZONE_PEDO'].str.replace('dekk/mbel', 'dekkMbel')
print("✅ Colonne 'ZONE_PEDO' créée et nettoyée.")

✅ Colonne 'ZONE_PEDO' créée et nettoyée.


In [8]:
display(gdf_parcellaire.head())

,Id,N°_PARCEL,N°_FOYER,SAISON,NOM_UTILIS,NOM_PROPRI,UTL_2012,NOM_CHAMPS,DIST_ENQ,DIST_CARTO,...,X_Centroid,Y_Centroid,Stock_C,StockC_0_3,parcel_id,Surface__1,Dist_Cen_1,geometry,Type_champ,ZONE_PEDO
0,0,147.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Pifind,1.0,0.0,...,337365.773977,1.603611e+06,9.22,17.44,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,3181.224210,17.652819,"MULTIPOLYGON (((337434.305 1603598.415, 337435...",CC,dior_cc_avec_arbr
1,0,147.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Pifind,1.0,0.0,...,337365.773977,1.603611e+06,9.22,17.44,c6c85252-0ee6-44b2-97bf-99724c6dd923,0.000000,0.000000,"POLYGON ((337438.043 1603638.866, 337438.614 1...",CC,dior_cc_sans_arbr
2,0,149.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Ngolsima,2.0,0.0,...,336099.653300,1.603497e+06,6.60,10.45,dae3c8e5-9f33-4354-826d-aed699b6c17c,1045.516931,46.730772,"MULTIPOLYGON (((336159.144 1603477.128, 336160...",CB,dekk_cb_avec_arbr
3,0,149.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Ngolsima,2.0,0.0,...,336099.653300,1.603497e+06,6.60,10.45,c33f3753-34ef-4b61-ba17-3cc4fe29bbd3,0.000000,0.000000,"POLYGON ((336165.254 1603521.577, 336167.671 1...",CB,dekk_cb_sans_arbr
4,0,150.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Maygueran,2.0,0.0,...,336302.866304,1.603039e+06,6.44,9.38,e85b0517-6fda-4422-95a3-571360932f23,2503.005594,8.629949,"MULTIPOLYGON (((336335.234 1602955.393, 336334...",CB,dior_cb_avec_arbr


In [9]:
# --- 5. SAUVEGARDE ---
output_parcellaire_enrichi_path.parent.mkdir(parents=True, exist_ok=True)
gdf_parcellaire.to_file(output_parcellaire_enrichi_path, driver='ESRI Shapefile')
print(f"\n✅ Shapefile du parcellaire enrichi sauvegardé dans :\n   {output_parcellaire_enrichi_path}")


✅ Shapefile du parcellaire enrichi sauvegardé dans :
   C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\shapefiles\processed\parcellaire_enrichi.shp
